# JRC European Power Plant Survey
**CH4Net v2 (retrained, div_factor=8) — Approach A spatial specificity across diverse European terrains**

Runs the live pipeline on each JRC site, then immediately computes the signal-to-control ratio at the plant coordinates vs a nearby rural reference crop.  
Results feed directly into the Section 6.2 table in the experimental document.

**Before running:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Cell 1: Mount Drive and clone/pull repo ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

REPO_DIR    = '/content/methane-api'
WEIGHTS     = '/content/drive/MyDrive/weights/ch4net_div8_retrained.pth'
RESULTS_DIR = '/content/drive/MyDrive/jrc_survey_results'   # persists to Drive
NPY_CACHE   = os.path.join(REPO_DIR, 'data/npy_cache')

os.makedirs(RESULTS_DIR, exist_ok=True)

# Clone repo if not present
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone',
        'https://github.com/vincentcorn2/methane-emission-tracker.git',
        REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Install deps
subprocess.run(['pip', 'install', '-q', 'rasterio', 'pyproj', 'scipy'], check=True)

print('Setup complete.')
print(f'Weights exist: {os.path.exists(WEIGHTS)}')

In [ ]:
# ── Cell 2: Copernicus credentials ─────────────────────────────────────────────
# These are used by the Copernicus API client inside the pipeline.
# Set them here so you don't have to touch any config files.
import os

os.environ['COPERNICUS_USER']     = 'your_email@example.com'   # ← fill in
os.environ['COPERNICUS_PASSWORD'] = 'your_password'            # ← fill in

print('Credentials set (not printed for security).')

In [ ]:
# ── Cell 3: JRC site list ──────────────────────────────────────────────────────
# Each entry: name → (lat, lon, fuel_type, country, terrain_description)
# Bounding box is auto-generated as 0.4°×0.4° centered on each site.
# Sites span 7 countries and 5 terrain types for maximum diversity.
# Groningen is included as a positive control (2.95× S/C in overnight_validation).

SITES = {
    # ── Already tested (for comparison baseline) ──────────────────────────────
    'groningen':      (53.252,   6.682, 'gas_field',  'NL', 'Positive control — TROPOMI confirmed emitter'),
    'eemshaven':      (53.437,   6.881, 'gas',        'NL', 'Flat coastal Netherlands — 1.38× baseline'),
    'emsland':        (52.481,   7.306, 'gas',        'DE', 'Flat inland Germany — suppressed baseline'),

    # ── Not yet tested — explicitly flagged in experimental doc ───────────────
    'irsching':       (48.767,  11.582, 'gas',        'DE', 'Bavaria hilly river valley'),
    'belchatow':      (51.263,  19.332, 'coal',       'PL', 'Central Poland flat plains — Europes largest coal plant'),

    # ── New diverse terrain sites ─────────────────────────────────────────────
    'maasvlakte':     (51.951,   4.004, 'coal',       'NL', 'Rotterdam port — coastal industrial delta'),
    'philippsburg':   (49.252,   8.459, 'gas',        'DE', 'Rhine plain — flat agricultural Germany'),
    'cordemais':      (47.275,  -1.874, 'coal',       'FR', 'Loire estuary — Atlantic coast France'),
    'brindisi':       (40.620,  17.850, 'coal',       'IT', 'Southern Italy Mediterranean coast'),
    'avedore':        (55.600,  12.450, 'gas',        'DK', 'Denmark flat Nordic coastal'),
    'pocerady':       (50.385,  13.539, 'coal',       'CZ', 'Bohemian plains Czech Republic'),
    'pembroke':       (51.682,  -4.990, 'gas',        'UK', 'Coastal Wales Atlantic'),
    'dunkerque':      (51.033,   2.377, 'mixed',      'FR', 'Northern France industrial coast'),
    'larobla':        (42.784,  -5.618, 'coal',       'ES', 'Northern Spain Cantabrian mountain valley'),
    'rugeley':        (52.762,  -1.897, 'coal',       'UK', 'English Midlands — inland flat'),
    'turow':          (51.004,  14.912, 'coal',       'PL', 'Sudeten foothills — Polish-German border'),
}

def make_polygon(lat, lon, delta=0.20):
    """0.4°×0.4° bounding box centered on lat/lon."""
    return (f"POLYGON(({lon-delta:.4f} {lat-delta:.4f}, "
            f"{lon+delta:.4f} {lat-delta:.4f}, "
            f"{lon+delta:.4f} {lat+delta:.4f}, "
            f"{lon-delta:.4f} {lat+delta:.4f}, "
            f"{lon-delta:.4f} {lat-delta:.4f}))")

print(f'Sites to survey: {len(SITES)}')
for name, (lat, lon, fuel, country, desc) in SITES.items():
    print(f'  {name:<14} {country}  {fuel:<9}  {desc}')

In [ ]:
# ── Cell 4: Approach A helper — signal/control ratio from GeoTIFF ──────────────
import numpy as np
import glob

try:
    import rasterio
    from rasterio.transform import rowcol
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print('WARNING: rasterio not available — install it in Cell 1')

CROP_PX   = 100   # half-width: 100px each side → 200×200 px = 2km×2km at 10m
CTRL_OFFSET_M = 5000   # control crop offset: 5 km inland (north)

def approach_a_from_geotiff(geotiff_path, site_lat, site_lon,
                             ctrl_offset_deg=0.045):  # ~5km north
    """
    Load probability GeoTIFF, extract 200×200 crop at (site_lat, site_lon)
    and a matched control crop offset northward by ctrl_offset_deg (~5km).
    Returns (site_mean, ctrl_mean, sc_ratio) or (None, None, None) on error.
    """
    if not HAS_RASTERIO:
        return None, None, None
    try:
        with rasterio.open(geotiff_path) as src:
            prob = src.read(1).astype(np.float32)   # full probability map
            H, W = prob.shape

            # Site pixel coordinates
            s_row, s_col = rowcol(src.transform, site_lon, site_lat)
            # Control: same lon, 5km north
            c_row, c_col = rowcol(src.transform, site_lon, site_lat + ctrl_offset_deg)

            def safe_crop(r, c):
                r0, r1 = r - CROP_PX, r + CROP_PX
                c0, c1 = c - CROP_PX, c + CROP_PX
                if r0 < 0 or r1 > H or c0 < 0 or c1 > W:
                    return None   # out of bounds
                return prob[r0:r1, c0:c1]

            site_crop = safe_crop(s_row, s_col)
            ctrl_crop = safe_crop(c_row, c_col)

            if site_crop is None or ctrl_crop is None:
                return None, None, None

            site_mean = float(site_crop.mean())
            ctrl_mean = float(ctrl_crop.mean())
            sc_ratio  = site_mean / ctrl_mean if ctrl_mean > 1e-9 else float('inf')
            return site_mean, ctrl_mean, sc_ratio

    except Exception as e:
        print(f'    GeoTIFF analysis error: {e}')
        return None, None, None


def find_latest_geotiff(site_output_dir):
    """Find the most recently written .tif in the site output directory."""
    tifs = glob.glob(os.path.join(site_output_dir, '*.tif'))
    return max(tifs, key=os.path.getmtime) if tifs else None


def sc_label(ratio):
    if ratio is None:    return '— (error)'
    if ratio >= 1.5:     return f'{ratio:.3f}  ✓✓ STRONG'
    if ratio >= 1.1:     return f'{ratio:.3f}  ✓ Signal'
    if ratio >= 0.9:     return f'{ratio:.3f}  ~ Neutral'
    return f'{ratio:.3f}  ✗ Suppressed'

print('Approach A helper ready.')

In [ ]:
# ── Cell 5: Run pipeline + Approach A for each site ────────────────────────────
# This cell runs sequentially. Each site takes ~45-90 min (download + GPU inference).
# Results are printed and saved to Drive as you go — safe to interrupt and resume.

import subprocess, json, time
from datetime import datetime

# Date range: use summer 2024 for low cloud cover across Europe
DATE_START = '2024-06-01'
DATE_END   = '2024-08-31'

all_results = []
results_json = os.path.join(RESULTS_DIR, 'survey_results.json')

# Load any previously completed results so you can resume if interrupted
if os.path.exists(results_json):
    with open(results_json) as f:
        all_results = json.load(f)
    done_sites = {r['site'] for r in all_results}
    print(f'Resuming — {len(done_sites)} sites already completed: {done_sites}')
else:
    done_sites = set()

print(f'\n{"="*70}')
print(f'JRC Survey — {len(SITES)} sites — CH4Net v2 weights')
print(f'{"="*70}\n')

for site_name, (lat, lon, fuel, country, desc) in SITES.items():
    if site_name in done_sites:
        print(f'[SKIP] {site_name} — already done')
        continue

    print(f'\n[{site_name.upper()}]  {desc}')
    print(f'  Coords: {lat:.3f}°N, {lon:.3f}°E  |  Fuel: {fuel}  |  {country}')

    polygon      = make_polygon(lat, lon)
    site_out_dir = os.path.join(RESULTS_DIR, site_name)
    os.makedirs(site_out_dir, exist_ok=True)

    cmd = [
        'python', '-m', 'scripts.live_pipeline',
        '--region',       polygon,
        '--start',        DATE_START,
        '--end',          DATE_END,
        '--weights',      WEIGHTS,
        '--max-products', '1',        # best cloud-free scene only
        '--max-cloud',    '20',
        '--threshold',    '0.18',
        '--output',       site_out_dir,
    ]

    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_DIR)
    elapsed = (time.time() - t0) / 60

    if result.returncode != 0:
        print(f'  ✗ Pipeline failed ({elapsed:.1f} min)')
        print('  STDERR:', result.stderr[-500:])   # last 500 chars
        row = dict(site=site_name, lat=lat, lon=lon, fuel=fuel, country=country,
                   desc=desc, status='PIPELINE_ERROR', site_mean=None,
                   ctrl_mean=None, sc_ratio=None, elapsed_min=round(elapsed,1))
    else:
        print(f'  Pipeline complete ({elapsed:.1f} min)')
        geotiff = find_latest_geotiff(site_out_dir)
        if geotiff is None:
            print(f'  ✗ No GeoTIFF found in {site_out_dir}')
            row = dict(site=site_name, lat=lat, lon=lon, fuel=fuel, country=country,
                       desc=desc, status='NO_GEOTIFF', site_mean=None,
                       ctrl_mean=None, sc_ratio=None, elapsed_min=round(elapsed,1))
        else:
            print(f'  GeoTIFF: {os.path.basename(geotiff)}')
            s_mean, c_mean, sc = approach_a_from_geotiff(geotiff, lat, lon)
            print(f'  Site mean:  {s_mean:.6f}' if s_mean else '  Site mean:  N/A')
            print(f'  Ctrl mean:  {c_mean:.6f}' if c_mean else '  Ctrl mean:  N/A')
            print(f'  S/C ratio:  {sc_label(sc)}')
            row = dict(site=site_name, lat=lat, lon=lon, fuel=fuel, country=country,
                       desc=desc, status='OK',
                       site_mean=round(s_mean, 6) if s_mean else None,
                       ctrl_mean=round(c_mean, 6) if c_mean else None,
                       sc_ratio=round(sc, 3) if sc else None,
                       geotiff=os.path.basename(geotiff),
                       elapsed_min=round(elapsed,1))

    all_results.append(row)
    done_sites.add(site_name)

    # Save after every site so a disconnect doesn't lose results
    with open(results_json, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'  Saved to Drive.')

print(f'\n{"="*70}')
print(f'Survey complete — {len(all_results)} sites processed')

In [ ]:
# ── Cell 6: Results table ──────────────────────────────────────────────────────
# Run this anytime to see current results, even mid-survey.

import json, os

results_json = os.path.join(RESULTS_DIR, 'survey_results.json')
with open(results_json) as f:
    all_results = json.load(f)

# Sort by S/C ratio descending
done = [r for r in all_results if r.get('sc_ratio') is not None]
failed = [r for r in all_results if r.get('sc_ratio') is None]
done.sort(key=lambda r: r['sc_ratio'], reverse=True)

print(f'CH4Net v2 — JRC European Survey — Approach A Results')
print(f'Baseline: Groningen 2.948 ✓✓ | Eemshaven 1.381 ✓ | Emsland 0.694 ✗')
print(f'{"─"*90}')
hdr = f'{"Site":<14} {"Country":<5} {"Fuel":<9} {"Site Mean":>10} {"Ctrl Mean":>10} {"S/C":>7}  Result'
print(hdr)
print(f'{"─"*90}')

for r in done:
    ratio = r['sc_ratio']
    if ratio >= 1.5:   label = '✓✓ STRONG'
    elif ratio >= 1.1: label = '✓ Signal'
    elif ratio >= 0.9: label = '~ Neutral'
    else:              label = '✗ Suppressed'
    print(f"{r['site']:<14} {r['country']:<5} {r['fuel']:<9} "
          f"{r['site_mean']:>10.6f} {r['ctrl_mean']:>10.6f} {ratio:>7.3f}  {label}")

if failed:
    print(f'\nFailed / no GeoTIFF:')
    for r in failed:
        print(f"  {r['site']:<14}  {r.get('status','?')}")

print(f'\n{"─"*90}')
if done:
    ratios = [r['sc_ratio'] for r in done]
    n_strong  = sum(1 for x in ratios if x >= 1.5)
    n_signal  = sum(1 for x in ratios if 1.1 <= x < 1.5)
    n_neutral = sum(1 for x in ratios if 0.9 <= x < 1.1)
    n_supp    = sum(1 for x in ratios if x < 0.9)
    print(f'Summary: {n_strong} STRONG  |  {n_signal} Signal  |  {n_neutral} Neutral  |  {n_supp} Suppressed')
    print(f'Mean S/C: {sum(ratios)/len(ratios):.3f}  |  Max: {max(ratios):.3f}  |  Min: {min(ratios):.3f}')

In [ ]:
# ── Cell 7 (OPTIONAL): Re-run Approach A on a site with a different control ────
# Use this if you want to test inland vs coastal control for any specific site.
# e.g. for Groningen: compare coastal control (original) vs inland agricultural control

RETEST_SITE     = 'groningen'
CTRL_OFFSET_DEG = 0.09    # ~10km north (inland, away from coast)

if RETEST_SITE in {r['site'] for r in all_results}:
    lat, lon = SITES[RETEST_SITE][:2]
    site_out_dir = os.path.join(RESULTS_DIR, RETEST_SITE)
    geotiff = find_latest_geotiff(site_out_dir)
    if geotiff:
        s_mean, c_mean, sc = approach_a_from_geotiff(
            geotiff, lat, lon, ctrl_offset_deg=CTRL_OFFSET_DEG)
        print(f'{RETEST_SITE} — inland control ({CTRL_OFFSET_DEG:.2f}° north ~{CTRL_OFFSET_DEG*111:.0f}km)')
        print(f'  Site mean:  {s_mean:.6f}')
        print(f'  Ctrl mean:  {c_mean:.6f}')
        print(f'  S/C ratio:  {sc_label(sc)}')
        print(f'\nOriginal (coastal control):  {[r for r in all_results if r["site"]==RETEST_SITE][0]["sc_ratio"]:.3f}')
        print(f'Inland control:              {sc:.3f}')
        print('If both are >1.5, the Groningen signal is robust to control placement.')
    else:
        print(f'No GeoTIFF found for {RETEST_SITE} — run Cell 5 first')
else:
    print(f'{RETEST_SITE} not yet processed')